# RDF Extractions

In [ ]:
import base64
import json
import os
from pathlib import Path

import requests
from PIL import Image

# 1. CONFIGURATION

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "YOUR_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY environment variable not set. Please set your API key.")

MODEL = "gemini-2.5-pro"
ENDPOINT = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent?key={GEMINI_API_KEY}"


# 2. FUNCTIONS
def encode_image_to_base64(image_path: str) -> str:
    try:
        with Image.open(image_path) as img:
            # Ensure image is RGB
            img = img.convert("RGB")
            # Read the bytes from the image
            buffer = Path(image_path).read_bytes()
            return base64.b64encode(buffer).decode("utf-8")
    except FileNotFoundError:
        print(f"Error: Image file not found at {image_path}")
        return None
    except Exception as e:
        print(f"An error occurred while processing the image: {e}")
        return None

def build_request(image_b64: str, prompt: str) -> dict:
    # JSON payload for the Gemini API multimodal request.
    return {
        "contents": [
            {
                "role": "user",
                "parts": [
                    {"inlineData": {"mimeType": "image/jpeg", "data": image_b64}},
                    {"text": prompt},
                ],
            }
        ],
        "generationConfig": {
            "temperature": 0.1,  # Lower temperature for more deterministic, structured output
            "topP": 0.9,
            "maxOutputTokens": 8192,  # Increased for potentially complex outputs
        },
    }

def call_gemini(payload: dict) -> dict:
    headers = {"Content-Type": "application/json"}
    try:
        response = requests.post(ENDPOINT, headers=headers, data=json.dumps(payload))
        response.raise_for_status()  # Raises an HTTPError for bad responses (4xx or 5xx)
        return response.json()
    except requests.exceptions.HTTPError as e:
        print(f"An HTTP error occurred: {e}")
        print(f"Response body: {e.response.text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None


# 3. MAIN EXECUTION
if __name__ == "__main__":
    img_path = "/content/table_1.1.jpg"
    output_filename = "output.json"

    prompt_text = """
    You are a multimodal knowledge-extraction agent.



    \### Objective



    Extract concepts and relationships from diagrams, flowcharts, or tables and generate RDF triples in Turtle syntax using SKOS and a custom namespace.



    \### Instructions



    1\. \*\*Prefixes (always declare):\*\*



    &nbsp;  ```turtle

    &nbsp;  @prefix she:  <https://soilwise-he.github.io/soil-health#> .

    &nbsp;  @prefix skos: <http://www.w3.org/2004/02/skos/core#> .

    &nbsp;  ```



    2\. \*\*Nodes (Concepts):\*\*



    &nbsp;  \* Mint a URI in the `she:` namespace, using \*\*PascalCase\*\* (initial capitalization).

    &nbsp;  \* Declare as `a skos:Concept`.

    &nbsp;  \* Add `skos:prefLabel` with exact text from the image, using \*\*all lowercase\*\*.

    &nbsp;  \* Optionally add `skos:definition` if the image shows definitions.



    3\. \*\*Edges (Relationships):\*\*



    &nbsp;  \* Use `skos:narrower` or `skos:broader` for hierarchical links.

    &nbsp;  \* Otherwise, if it expresses a semantic relation that goes beyond SKOS, define a custom property in \*\*camelCase\*\* using clear natural language (e.g., `she:measures`, `she:affects`). Ensure your custom property name clearly reflects its meaning.

    &nbsp;  \* Custom properties must follow ontology property conventions and use the `she:` namespace.



    4\. \*\*Output Requirements:\*\*



    &nbsp;  \* Valid Turtle syntax only.

    &nbsp;  \* One triple per line, ending with a period.

    &nbsp;  \* Group all triples for the same subject with semicolons.

    &nbsp;  \* Do not use any other prefixes or ontologies besides `skos:` and `she:`.



    5\. Output \*\*only\*\* valid Turtle syntax:



    &nbsp;  \* One triple per line, ending with a period.

    &nbsp;  \* Group all triples for the same subject together, separated by semicolons.

    &nbsp;  \* Do not include any other prefixes or ontologies.



    \### Example Output Skeleton



    ```turtle

    @prefix she:  <https://soilwise-he.github.io/soil-health#> .

    @prefix skos: <http://www.w3.org/2004/02/skos/core#> .



    she:TopConcept a skos:Concept ;

    &nbsp;   skos:prefLabel "top concept" ;

    &nbsp;   skos:narrower she:ChildConceptA,

    &nbsp;                 she:ChildConceptB ;

    &nbsp;   skos:definition "Top concept ..." .



    she:ChildConceptA a skos:Concept ;

    &nbsp;   skos:prefLabel "child concept a" ;

    &nbsp;   she:myCustomRelation she:OtherConcept .



    she:ChildConceptB a skos:Concept ;

    &nbsp;   skos:prefLabel "child concept b" .

    ```



    """

    print(f"Processing image: {img_path}")
    image_b64 = encode_image_to_base64(img_path)

    if image_b64:
        request_body = build_request(image_b64, prompt_text)
        print("Sending request to Gemini API...")
        result = call_gemini(request_body)

        if result and "candidates" in result:
            extracted_rdf = result["candidates"][0]["content"]["parts"][0]["text"]

            if extracted_rdf.strip().startswith("```turtle"):
                extracted_rdf = extracted_rdf.strip()[len("```turtle"):-len("```")].strip()

            print("\n--- Gemini RDF Output ---")
            print(extracted_rdf)

            # Prepare the output JSON data
            output_data = {
                "dataset": [
                    {
                        "table": img_path,
                        "rdf_graph_turtle": extracted_rdf
                    }
                ]
            }

            # Save the output as a JSON file
            with open(output_filename, "w", encoding="utf-8") as f:
                json.dump(output_data, f, indent=4)
            print(f"\n--- Successfully saved to {output_filename} ---")
        else:
            print("Failed to get a valid response from the Gemini API.")


Processing image: /content/table_1.1.jpg
Sending request to Gemini API...

--- Gemini RDF Output ---
@prefix she:  <https://soilwise-he.github.io/soil-health#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .

she:SocietalNeeds a skos:Concept ;
    skos:prefLabel "societal needs" ;
    skos:narrower she:Biomass,
                  she:Water,
                  she:Climate,
                  she:Biodiversity,
                  she:Infrastructure .

she:SoilServices a skos:Concept ;
    skos:prefLabel "soil services" .

she:SoilThreatIndicators a skos:Concept ;
    skos:prefLabel "soil threat indicators" ;
    skos:narrower she:SoilOrganicCarbon,
                  she:SoilNutrientStatus,
                  she:SoilAcidification,
                  she:SoilPollution,
                  she:SoilBiodiversity,
                  she:SoilErosion,
                  she:SoilCompaction,
                  she:SoilSealing .

she:Biomass a skos:Concept ;
    skos:prefLabel "biomass" ;
    skos:br